In [2]:
import os
import unicodedata # instead of referencing a local file
print(unicodedata.unidata_version)
import pandas as pd

from google.colab import drive, files
from IPython.display import display

15.0.0


In [11]:
# Choose where to save the database:
# "drive"    = save to Google Drive
# "computer" = save to the Colab runtime first,
#              then download it to your computer

output_source = "drive"


# Set output path

if output_source == "drive":

    drive.mount("/content/drive")

    save_path = (
    "/content/drive/MyDrive/Character Complexity/unicode_character_database.pkl"
)

elif output_source == "computer":

    save_path = (
        "/content/"
        "unicode_character_database.pkl"
    )

else:

    raise ValueError(
        "output_source must be 'drive' or 'computer'."
    )


print("Database will be saved to:")
print(save_path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Database will be saved to:
/content/drive/MyDrive/Character Complexity/unicode_character_database.pkl


In [5]:
# Create Unicode character database
#
# Based on alphabet_complexity by McCullen.
#
# The database contains all Unicode characters in these categories:
#   L = Letters
#   N = Numbers
#   P = Punctuation
#   S = Symbols
#
# The following whitespace characters are also included because they
# are relevant to text parsing and will later be assigned complexity = 0:
#   U+0020 SPACE
#   U+0009 TAB
#   U+000A NEWLINE
#
# Combining marks (M) are intentionally excluded.
#
# NFC normalization is applied to input text before character analysis.
# The Unicode database itself contains individual Unicode codepoints
# and is not normalized.


def create_unicode_database():

    included_categories = {
        "L",
        "N",
        "P",
        "S"
    }

    records = []

    # Add Unicode letters, numbers, punctuation, and symbols.

    for codepoint in range(0x110000): # the entire Unicode code-space

        char = chr(codepoint)
        category = unicodedata.category(char)

        if category[0] not in included_categories:
            continue

        records.append({
            "codepoint": codepoint,
            "char": char,
            "unicode": f"U+{codepoint:04X}",
            "unicode_name": unicodedata.name(char, None), # if no name, insert "None", keep character if there is a codepoint for it
            "category": category
        })


    # Add whitespace characters required for text parsing.
    #
    # These are intentionally included even though their Unicode
    # categories are not L, N, P, or S.

    whitespace_characters = {
        " ": "SPACE",
        "\t": "TAB",
        "\n": "NEWLINE"
    }

    for char, name in whitespace_characters.items():

        codepoint = ord(char)

        records.append({
            "codepoint": codepoint,
            "char": char,
            "unicode": f"U+{codepoint:04X}",
            "unicode_name": name,
            "category": unicodedata.category(char)
        })


    return pd.DataFrame(records)

In [6]:
# Create database

df_unicode = create_unicode_database()


print(
    f"Unicode characters in database: "
    f"{len(df_unicode):,}"
)

print(
    f"Columns: "
    f"{len(df_unicode.columns)}"
)

print()
print("First 20 rows:")

display(
    df_unicode.head(20)
)


# NFC validation
#
# NFC normalization is applied to input text before lookup.
# The database itself contains individual Unicode codepoints.
#
# These checks confirm that the explicitly added whitespace
# characters are unchanged by NFC.

print()
print("NFC validation")

for char in [
    " ",
    "\t",
    "\n"
]:

    normalized = unicodedata.normalize(
        "NFC",
        char
    )

    status = (
        "OK"
        if char == normalized
        else "CHANGED"
    )

    print(
        repr(char),
        "→",
        repr(normalized),
        status
    )


# Final database summary

print()
print("Final database summary")

print(
    f"Rows: "
    f"{len(df_unicode):,}"
)

print(
    f"Columns: "
    f"{len(df_unicode.columns)}"
)

print(
    f"Unique characters: "
    f"{df_unicode['char'].nunique():,}"
)

print(
    f"Unique codepoints: "
    f"{df_unicode['codepoint'].nunique():,}"
)




Unicode characters in database: 146,550
Columns: 5

First 20 rows:


,codepoint,char,unicode,unicode_name,category
0,33,!,U+0021,EXCLAMATION MARK,Po
1,34,"""",U+0022,QUOTATION MARK,Po
2,35,#,U+0023,NUMBER SIGN,Po
3,36,$,U+0024,DOLLAR SIGN,Sc
4,37,%,U+0025,PERCENT SIGN,Po
5,38,&,U+0026,AMPERSAND,Po
6,39,',U+0027,APOSTROPHE,Po
7,40,(,U+0028,LEFT PARENTHESIS,Ps
8,41,),U+0029,RIGHT PARENTHESIS,Pe
9,42,*,U+002A,ASTERISK,Po



NFC validation
' ' → ' ' OK
'\t' → '\t' OK
'\n' → '\n' OK

Final database summary
Rows: 146,550
Columns: 5
Unique characters: 146,550
Unique codepoints: 146,550


In [7]:
# Validate database

print()
print("Unicode database validation")
print()


expected_columns = [
    "codepoint",
    "char",
    "unicode",
    "unicode_name",
    "category"
]

missing_columns = [
    column
    for column in expected_columns
    if column not in df_unicode.columns
]

if missing_columns:

    print(
        "ERROR: Missing columns:",
        missing_columns
    )

else:

    print("Columns: OK")


# Check duplicate characters

duplicate_characters = (
    df_unicode["char"]
    .duplicated()
    .sum()
)

print(
    f"Duplicate characters: "
    f"{duplicate_characters}"
)



# Check duplicate codepoints

duplicate_codepoints = (
    df_unicode["codepoint"]
    .duplicated()
    .sum()
)

print(
    f"Duplicate codepoints: "
    f"{duplicate_codepoints}"
)

# Check that only intended Unicode categories are present
# The database intentionally includes:
#   L = Letters
#   N = Numbers
#   P = Punctuation
#   S = Symbols
#
# In addition, three whitespace characters are deliberately
# added:
#   U+0020 SPACE     -> Zs
#   U+0009 TAB       -> Cc
#   U+000A NEWLINE   -> Cc
#
# No other Z, C, or M characters should be present.

allowed_category_prefixes = {
    "L",
    "N",
    "P",
    "S"
}

intentional_whitespace = {
    " ",
    "\t",
    "\n"
}

unexpected_categories = df_unicode[
    (
        ~df_unicode["category"]
        .str[0]
        .isin(allowed_category_prefixes)
    )
    &
    (
        ~df_unicode["char"]
        .isin(intentional_whitespace)
    )
]

print(
    f"Unexpected categories: "
    f"{len(unexpected_categories):,}"
)

if len(unexpected_categories) > 0:

    print()
    print(
        "Examples of unexpected categories:"
    )

    display(
        unexpected_categories[
            [
                "codepoint",
                "char",
                "unicode",
                "unicode_name",
                "category"
            ]
        ].head(20)
    )



# Check for Unicode characters without names
# Characters with Unicode codepoints are retained even if they lack a unicode name: "none" will be inserted for characters without unicode names

# A valid Unicode codepoint does not guarantee every font can render the character, rendering depends on font coverage
# Some characters may not be displayable in the notebook and may appear as rectangles if the notebook font lacks a glyph for them


missing_names = df_unicode[
    df_unicode["unicode_name"].isna()
]

print(
    f"Characters without Unicode names: "
    f"{len(missing_names):,}"
)

if len(missing_names) > 0:

    print()
    print(
        "Examples of characters without names:"
    )

    display(
        missing_names[
            [
                "codepoint",
                "char",
                "unicode",
                "unicode_name",
                "category"
            ]
        ].head(20)
    )


# Check category distribution

print()
print("Unicode category distribution")

display(
    df_unicode[
        "category"
    ]
    .value_counts()
    .sort_index()
)


# Check whitespace

print()
print("Whitespace checks")

for char, label in [
    (" ", "space"),
    ("\t", "tab"),
    ("\n", "newline")
]:

    matches = df_unicode[
        df_unicode["char"] == char
    ]

    print(
        f"{label}: "
        f"{len(matches)} entry"
    )


# Check representative characters

print()
print("Representative character checks")

examples = [
    "A",
    "Ж",
    "中",
    "अ",
    "م",
    "Δ",
    "$",
    "&"
]

example_table = df_unicode[
    df_unicode["char"].isin(
        examples
    )
]

display(
    example_table[
        [
            "char",
            "unicode",
            "unicode_name",
            "category"
        ]
    ]
)


# Check for duplicate codepoints and unexpected categories again after adding whitespace

if duplicate_characters != 0:

    print(
        "WARNING: Duplicate characters found."
    )

if duplicate_codepoints != 0:

    print(
        "WARNING: Duplicate codepoints found."
    )

if len(unexpected_categories) != 0:

    print(
        "WARNING: Unexpected Unicode categories found."
    )


Unicode database validation

Columns: OK
Duplicate characters: 0
Duplicate codepoints: 0
Unexpected categories: 0
Characters without Unicode names: 6,145

Examples of characters without names:


,codepoint,char,unicode,unicode_name,category
62566,94208,𗀀,U+17000,None,Lo
62567,94209,𗀁,U+17001,None,Lo
62568,94210,𗀂,U+17002,None,Lo
62569,94211,𗀃,U+17003,None,Lo
62570,94212,𗀄,U+17004,None,Lo
62571,94213,𗀅,U+17005,None,Lo
62572,94214,𗀆,U+17006,None,Lo
62573,94215,𗀇,U+17007,None,Lo
62574,94216,𗀈,U+17008,None,Lo
62575,94217,𗀉,U+17009,None,Lo



Unicode category distribution


,count
category,
Cc,2
Ll,2233
Lm,397
Lo,131612
Lt,31
Lu,1831
Nd,680
Nl,236
No,915



Whitespace checks
space: 1 entry
tab: 1 entry
newline: 1 entry

Representative character checks


,char,unicode,unicode_name,category
3,$,U+0024,DOLLAR SIGN,Sc
5,&,U+0026,AMPERSAND,Po
32,A,U+0041,LATIN CAPITAL LETTER A,Lu
728,Δ,U+0394,GREEK CAPITAL LETTER DELTA,Lu
857,Ж,U+0416,CYRILLIC CAPITAL LETTER ZHE,Lu
1311,م,U+0645,ARABIC LETTER MEEM,Lo
1794,अ,U+0905,DEVANAGARI LETTER A,Lo
17659,中,U+4E2D,CJK UNIFIED IDEOGRAPH-4E2D,Lo


In [12]:
# Save database

if os.path.exists(save_path):

    print()
    print(
        f"A file already exists at:\n"
        f"{save_path}"
    )

    while True:

        response = input(
            "Overwrite existing file, choose a new filename, "
            "or cancel? [o/n/c]: "
        ).lower().strip()

        if response == "o":

            # Overwrite existing file
            df_unicode.to_pickle(save_path)

            print()
            print("Database overwritten:")
            print(save_path)

            break


        elif response == "n":

            # Ask for a new filename
            new_filename = input(
                "Enter a new filename "
                "(including .pkl): "
            ).strip()

            if not new_filename:

                print(
                    "Filename cannot be empty."
                )
                continue

            if not new_filename.endswith(".pkl"):

                new_filename += ".pkl"

            new_save_path = os.path.join(
                os.path.dirname(save_path),
                new_filename
            )

            # Check whether the new filename also exists
            if os.path.exists(new_save_path):

                print()
                print(
                    f"That file already exists:\n"
                    f"{new_save_path}"
                )

                continue

            save_path = new_save_path

            df_unicode.to_pickle(save_path)

            print()
            print("Database saved:")
            print(save_path)

            break


        elif response == "c":

            print()
            print("Database not saved.")

            break


        else:

            print(
                "Please enter 'o' for overwrite, "
                "'n' for new filename, or 'c' to cancel."
            )


else:

    df_unicode.to_pickle(save_path)

    print()
    print("Database saved:")
    print(save_path)


# Download database if saving to computer

if (
    output_source == "computer"
    and os.path.exists(save_path)
):

    print()
    print("Downloading database...")

    files.download(save_path)


Database saved:
/content/drive/MyDrive/Character Complexity/unicode_character_database.pkl
